# autoresearch x-DDPM — Experiment Analysis

Reads `results.tsv` and plots:
- `progress.png`: val_loss over time (kept=green, discarded=grey)
- `memory.png`: peak VRAM per experiment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
df = pd.read_csv("results.tsv", sep="\t")
print(f"Total experiments: {len(df)}")
print(
    f"Keep: {(df.status == 'keep').sum()}, Discard: {(df.status == 'discard').sum()}, Crash: {(df.status == 'crash').sum()}"
)
keep_rate = (df.status == "keep").mean() * 100
print(f"Keep rate: {keep_rate:.1f}%")
print(df.to_string())

In [ ]:
# --- Progress chart ---

fig, ax = plt.subplots(figsize=(14, 6))

kept = df[df.status == "keep"]
discarded = df[df.status != "keep"]
crashed = df[df.status == "crash"]

x = range(len(df))

# Plot all experiments with their val_loss
colors = df.status.map({"keep": "#2ca02c", "discard": "#aaaaaa", "crash": "#d62728"})
ax.scatter(x, df.val_loss, c=colors, zorder=3, s=60)

# Step line: running best (ignoring crashes)
valid = df[df.val_loss > 0].copy()
valid_x = list(valid.index)
running_best = []
best = float("inf")
for i, row in df.iterrows():
    if row.val_loss > 0 and row.status == "keep":
        best = min(best, row.val_loss)
    running_best.append(best if best < float("inf") else None)

# Fill in None values for step line
best_vals = []
b = float("inf")
for i, row in df.iterrows():
    if row.val_loss > 0 and row.status == "keep":
        b = min(b, row.val_loss)
    best_vals.append(b if b < float("inf") else float("nan"))

ax.step(
    x,
    best_vals,
    where="post",
    color="#2ca02c",
    linewidth=2,
    alpha=0.7,
    label="Running best",
)

# Label kept experiments
for i, row in df.iterrows():
    if row.status == "keep":
        ax.annotate(
            row.description,
            xy=(i, row.val_loss),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=7,
            color="#2ca02c",
            rotation=15,
        )

ax.set_xlabel("Experiment #")
ax.set_ylabel("val_loss (denoising MSE — lower is better)")
ax.set_title("x-DDPM autoresearch — val_loss over experiments")

# Legend
keep_patch = mpatches.Patch(color="#2ca02c", label="keep")
discard_patch = mpatches.Patch(color="#aaaaaa", label="discard")
crash_patch = mpatches.Patch(color="#d62728", label="crash")
ax.legend(handles=[keep_patch, discard_patch, crash_patch], loc="upper right")

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("progress.png", dpi=150)
plt.show()
print("Saved progress.png")

In [ ]:
# --- Memory chart ---

fig, ax = plt.subplots(figsize=(14, 5))

bar_colors = df.status.map(
    {"keep": "#2ca02c", "discard": "#aaaaaa", "crash": "#d62728"}
)
ax.bar(x, df.memory_gb, color=bar_colors, alpha=0.8)

ax.set_xlabel("Experiment #")
ax.set_ylabel("Peak VRAM (GB)")
ax.set_title("x-DDPM autoresearch — Peak VRAM per experiment")
ax.axhline(y=24, color="red", linestyle="--", alpha=0.5, label="24 GB limit (RTX 4090)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("memory.png", dpi=150)
plt.show()
print("Saved memory.png")

In [ ]:
# --- Summary stats ---

kept_df = df[df.status == "keep"]
if len(kept_df) > 0:
    baseline_loss = kept_df.iloc[0].val_loss
    best_loss = kept_df.val_loss.min()
    improvement = (baseline_loss - best_loss) / baseline_loss * 100
    print(f"\nBaseline val_loss:  {baseline_loss:.6f}")
    print(f"Best val_loss:      {best_loss:.6f}")
    print(f"Total improvement:  {improvement:.1f}%")

    print("\nTop 5 kept experiments by val_loss:")
    print(
        kept_df.nsmallest(5, "val_loss")[
            ["commit", "val_loss", "memory_gb", "description"]
        ].to_string(index=False)
    )

    # Delta from previous kept experiment
    kept_df = kept_df.copy()
    kept_df["delta"] = kept_df.val_loss.diff()
    kept_df["delta_pct"] = kept_df.delta / kept_df.val_loss.shift(1) * 100
    print("\nKept experiments with improvement delta:")
    print(
        kept_df[["commit", "val_loss", "delta", "delta_pct", "description"]].to_string(
            index=False
        )
    )